In [3]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install --no-cache-dir --force-reinstall transformers==4.41.2
!pip install accelerate bitsandbytes datasets peft trl

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 135.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 254.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 286.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 262.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 359.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 422.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 380.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 345.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 425.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 416.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 405.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 147.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.4.0
    Uninstalling fsspec-2026.4.0:
      Successfully uninstalled fsspec-2026.4.0
  Attempting uninstall: huggingface_hub


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import pipeline
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from peft import PeftModel
from trl import SFTTrainer, SFTConfig
import torch
import os
import json
from google.colab import userdata
from google import genai
from google.genai import types
import pandas as pd
from google.colab import drive
import torchvision
import shutil

In [2]:
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
HF_TOKEN = userdata.get('HF_TOKEN')
DATASET_PATH = "/content/Sample_1536.jsonl"
MAX_LENGTH = 1536 #Capped at 1536 due to consumer constraints, this value should be changed for a new data set!


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

def formatting_prompts_func(examples):
    texts = []
    for instr, out in zip(examples["instruction"], examples["output"]):
        messages = [
            {"role": "system", "content": "You are a senior Thermodynamics Engineer. Provide clear, accurate answers. Use step-by-step derivations with LaTeX when the question requires mathematical detail."},
            {"role": "user", "content": instr},
            {"role": "assistant", "content": out},
        ]
        texts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
        )
    return {"text": texts}

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

dataset = dataset.train_test_split(test_size=0.1)
train_data = dataset["train"]
val_data = dataset["test"]

sft_config = SFTConfig(
    output_dir="./Mistral-7B-Model-Checkpoint",
    dataset_text_field="text",
    max_length=MAX_LENGTH,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=5e-5,
    num_train_epochs=2,
    optim="paged_adamw_8bit",
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_steps=100,
    warmup_ratio=0.1,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=sft_config,
)

trainer.train()

os.makedirs("./Mistral-7B-Model", exist_ok=True)
with open("./Mistral-7B-Model/training_log.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

trainer.save_model("./Mistral-7B-Model-final")
print("Model saved.")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Adding EOS to eval dataset:   0%|          | 0/181 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/181 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
25,0.910947,0.822847,0.768570,727522.000000,0.781196
50,0.767892,0.765374,0.765028,1453177.000000,0.790554
75,0.759010,0.743878,0.734134,2179312.000000,0.793884
100,0.737476,0.736598,0.725277,2903854.000000,0.794887
102,0.737476,0.736569,0.725296,2952686.000000,0.794692


Model saved.


In [ ]:
shutil.make_archive("Mistral-7B-V2", 'zip', "./Mistral-7B-Model-final")

files.download("Mistral-7B-V2.zip")

files.download("./Mistral-7B-Model/training_log.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

MISTRAL_ADAPTER = "/content/drive/MyDrive/Mistral-7B-V2"
base = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
model = PeftModel.from_pretrained(base, MISTRAL_ADAPTER)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [3]:
# Performance Evaluation

from google.colab import userdata

drive.mount('/content/drive')

MERGED_PATH = "/content/drive/MyDrive/Mistral-7B-V2"
BASE_MODEL  = MODEL_ID
hf_token = userdata.get('HF_TOKEN')
GEMINI_KEY  = userdata.get('Gemin_Key')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# ── Evaluation questions Created by Gemini/Claude AI ────────────────────────────────────────────────────────────

EVAL_QUESTIONS = [
    {
        "id": "Q01", "topic": "Specific Enthalpy Calculation",
        "question": "Steam at a certain state has a specific internal energy of 2900 kJ/kg, a pressure of 500 kPa, and a specific volume of 0.4 m³/kg. Calculate the specific enthalpy of the steam.",
        "reference": "h = u + Pv = 2900 + (500 * 0.4) = 2900 + 200 = 3100 kJ/kg."
    },
    {
        "id": "Q02", "topic": "Enthalpy Change - Heating",
        "question": "Air is heated at constant pressure from 300 K to 600 K. Using cp = 1.005 kJ/(kg·K), calculate the specific enthalpy change. Explain why enthalpy is the appropriate property for constant pressure processes.",
        "reference": "Δh = cp * ΔT = 1.005 * (600 - 300) = 1.005 * 300 = 301.5 kJ/kg. At constant pressure the heat transfer equals the enthalpy change: Q = ΔH. Enthalpy accounts for both internal energy change and flow work (PΔv)."
    },
    {
        "id": "Q03", "topic": "Heat Engine Net Work",
        "question": "A heat engine operates between a furnace at 900 K and a cooling water reservoir at 300 K. In each cycle it absorbs 600 kJ from the furnace. Calculate the maximum work output and the heat rejected to the cooling water.",
        "reference": "Carnot efficiency = 1 - T_L/T_H = 1 - 300/900 = 0.667. Max work = η * Q_H = 0.667 * 600 = 400 kJ. Heat rejected = Q_H - W = 600 - 400 = 200 kJ."
    },
    {
        "id": "Q04", "topic": "Thermal Efficiency Comparison",
        "question": "Two heat engines operate between the same reservoirs at 800 K and 300 K. Engine A has a thermal efficiency of 40% and Engine B has a thermal efficiency of 65%. Which engine violates the second law of thermodynamics and why?",
        "reference": "Carnot efficiency = 1 - 300/800 = 0.625 = 62.5%. Engine A (40%) is below Carnot — possible and irreversible. Engine B (65%) exceeds Carnot efficiency — this violates the second law and is impossible."
    },
    {
        "id": "Q05", "topic": "Heat Engine Cycle Analysis",
        "question": "A heat engine receives 1500 kJ of heat per cycle and produces a net work output of 600 kJ. Calculate the thermal efficiency and the heat rejected. If the engine operates between reservoirs at 1000 K and 400 K, is the engine reversible, irreversible, or impossible?",
        "reference": "Efficiency = W/Q_H = 600/1500 = 0.40 = 40%. Heat rejected = Q_H - W = 1500 - 600 = 900 kJ. Carnot efficiency = 1 - 400/1000 = 0.60 = 60%. Since actual (40%) < Carnot (60%) the engine is irreversible but possible."
    },
]


def load_model(model_path, hf_token, label):
    is_finetuned = (label == "finetuned")
    print(f"  Loading {'finetuned (base + adapter)' if is_finetuned else 'base'}...")

    tokenizer = AutoTokenizer.from_pretrained(
        "mistralai/Mistral-7B-Instruct-v0.2", token=hf_token
    )
    tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        "mistralai/Mistral-7B-Instruct-v0.2",
        quantization_config=bnb_config,
        device_map="auto",
        token=hf_token,
    )

    if is_finetuned:
        model = PeftModel.from_pretrained(base, model_path)
    else:
        model = base

    model.eval()
    return model, tokenizer


def generate(model, tokenizer, question, max_new_tokens=1200):
    messages = [
        {"role": "system", "content": "You are a senior Thermodynamics Engineer. Provide clear, accurate answers with step-by-step reasoning."},
        {"role": "user",   "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def gemini_judge(question, reference, response, gemini_client):
    prompt = f"""You are a thermodynamics teacher scoring your student's answer.
    \n\nQUESTION: {question}
    \n\nREFERENCE ANSWER: {reference}
    \n\nSTUDENT RESPONSE: {response}
    \n\nScore the student response on the following criteria.Return ONLY a JSON object with these exact keys:\n\n
    {{\n  \"correctness\": <0-4, is the core answer/calculation correct? Good steps but wrong answers get partial scores.>,
    \n  \"Formula and Formatting\":0-2, Is the answer properly structured with formulas where applicable, does the student speak like a thermodynamic specialist?>,
    \n  \"reasoning\": <0-2, is the step-by-step reasoning sound? Deduct 1 if the student fails to present their answers in good format. Give a partial score for partially correct answers.>,
    \n  \"clarity\": <0-1, is the answer clearly explained?>,\n  \"hallucination\": <0 or -1, deduct 1 if the response contains confidently stated incorrect facts>,
    \n  \"total\": <sum of above, minimum 0>,\n  \"comment\": \"<one sentence explaining the main strength or weakness>\"\n}}\n"""

    try:
        response_obj = gemini_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.1,
            )
        )
        scores = json.loads(response_obj.text)
        scores["total"] = max(0, sum(
            scores.get(k, 0) for k in ["correctness", "reasoning", "clarity", "hallucination"]
        ))
        return scores
    except Exception as e:
        print(f"  ⚠️ Judge error: {e}")
        return {"correctness": 0, "reasoning": 0, "clarity": 0, "hallucination": 0, "total": 0, "comment": "Judge failed"}



gemini_client = genai.Client(api_key=GEMINI_KEY)
results = []

for label, path in [("finetuned", MERGED_PATH), ("base", BASE_MODEL)]:
    print(f"\n{'='*55}\n  Evaluating: {label.upper()}\n{'='*55}")

    model, tokenizer = load_model(path, hf_token, label)

    for q in EVAL_QUESTIONS:
        print(f"  ▶ {q['id']} — {q['topic']}")
        response = generate(model, tokenizer, q["question"])
        scores = gemini_judge(q["question"], q["reference"], response, gemini_client)
        print(f"     Judge score: {scores['total']}/9 | {scores['comment']}")
        results.append(
            {
            "model":         label,
            "id":            q["id"],
            "topic":         q["topic"],
            "response":      response,
            "correctness":   scores.get("correctness", 0),
            "reasoning":     scores.get("reasoning", 0),
            "clarity":       scores.get("clarity", 0),
            "hallucination": scores.get("hallucination", 0),
            "total":         scores["total"],
            "comment":       scores.get("comment", ""),
        }
        )

    del model, tokenizer
    torch.cuda.empty_cache()


df = pd.DataFrame(results)
summary = df.groupby("model")["total"].agg(["mean", "std", "min", "max"]).round(2)

print("\n" + "="*55)
print("FINAL SCORES (Gemini-as-Judge, max 9)")
print("="*55)
print(summary.to_string())

print("\nPer question:")
print(df.pivot(index="id", columns="model", values="total").to_string())

print("\nHallucination count:")
print(df.groupby("model")["hallucination"].apply(lambda x: (x < 0).sum()))

df.to_csv("results_2.csv", index=False)
print("\n✅ Saved")

Mounted at /content/drive

  Evaluating: FINETUNED
  Loading finetuned (base + adapter)...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

  ▶ Q01 — Refrigerator COP


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


     Judge score: 3/9 | The student correctly calculated the heat removed from the cold space but made a fundamental error in applying the First Law of Thermodynamics to calculate the heat rejected to the surroundings.
  ▶ Q03 — Free Expansion
     Judge score: 0.5/9 | The student correctly identified Q=0 and the final pressure value, but fundamentally misunderstood the work done in a free expansion and made incorrect assumptions about mass and constant pressure during expansion.
  ▶ Q04 — Throttling Process
     Judge score: 0.5/9 | The student fundamentally misunderstands the behavior of enthalpy and temperature during a throttling process, stating incorrect definitions and thermodynamic principles, though the explanation for its use in a refrigeration cycle is partially correct.

  Evaluating: BASE
  Loading base...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ▶ Q01 — Refrigerator COP
     Judge score: 2.5/9 | The student correctly identifies the formulas for COP and conservation of energy but fundamentally misinterprets the given electrical work as something to be divided by COP, leading to incorrect final answers.
  ▶ Q03 — Free Expansion
     Judge score: 0/9 | The student incorrectly applies the First Law of Thermodynamics and makes several fundamental errors in calculating both the final pressure and temperature, including assuming the amount of substance doubles and that pressure remains constant.
  ▶ Q04 — Throttling Process
     Judge score: 1.5/9 | The student fundamentally misunderstands the nature of a throttling process, incorrectly stating it is isentropic, that enthalpy decreases, and misapplying thermodynamic formulas and concepts.

FINAL SCORES (Gemini-as-Judge, max 9)
           mean   std  min  max
model                          
base       1.33  1.26  0.0  2.5
finetuned  1.33  1.44  0.5  3.0

Per question:
model  base  f